# 24 — MCP Server for ServiceTitan Tools

**Why this matters at ServiceTitan**: both the Senior DS (Onboarding) and Lead
DS (Operations/Finance) reqs name **MCP** ("Model Context Protocol") as a
preferred skill in 2026. MCP is the open protocol from Anthropic for connecting
LLMs to external tools and data — it's how Atlas could expose every
ServiceTitan API surface as a callable tool without bespoke wrappers per
client.

This notebook builds a working MCP server **from scratch** so you understand
the wire protocol, then shows how the same tools plug into the LangGraph agent
from notebook 11. We don't take a hard dependency on the `mcp` Python SDK —
the spec is small enough to implement directly, which is the right move for
interview prep where you may have to explain what's actually happening over
the wire.

## What we'll build

| Section | Topic |
|---------|-------|
| 1 | MCP protocol primer — JSON-RPC 2.0 over stdio/SSE |
| 2 | Minimal MCP server: tool registration + dispatch |
| 3 | ServiceTitan tools: jobs, technicians, customers, revenue |
| 4 | Schema validation + error handling |
| 5 | Multi-tenant auth and isolation |
| 6 | Wiring to a LangGraph agent (as a tool-use loop) |
| 7 | Production checklist: deployment, observability, rate limits |

References:
- Anthropic MCP spec: https://modelcontextprotocol.io/specification
- Python SDK: `pip install mcp` (we mention but don't use)


In [ ]:
import json
import uuid
import time
from dataclasses import dataclass, field
from typing import Callable, Any
from collections import defaultdict


---
## Section 1 — MCP Protocol Primer

MCP is **JSON-RPC 2.0** over a transport (stdio for local subprocess,
HTTP+SSE for remote, or WebSocket). The wire format is plain JSON.

### The four message types you need

| Type | Direction | Purpose |
|------|-----------|---------|
| `request` | client → server | invoke a method, expects a response |
| `response` | server → client | result or error for a prior request |
| `notification` | either way | fire-and-forget event |
| `error` | server → client | structured error response |

### The handshake

```
1. Client sends:  initialize { client info, protocol version }
2. Server sends:  response   { server info, capabilities }
3. Client sends:  notifications/initialized
4. Now ready: client can call tools/list, tools/call, resources/list, etc.
```

### The two endpoints we care about for tool use

| Method | Returns |
|--------|---------|
| `tools/list` | array of `{name, description, inputSchema}` |
| `tools/call` | `{content: [{type: "text"\|"json", text/data: ...}]}` |

That's basically all you need to expose ST data to an LLM via MCP.


In [ ]:
# Example MCP request/response over the wire — what bytes actually move:

example_request = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/call",
    "params": {
        "name": "get_jobs",
        "arguments": {"tenant_id": "acme-hvac", "status": "scheduled", "limit": 10},
    },
}

example_response = {
    "jsonrpc": "2.0",
    "id": 1,
    "result": {
        "content": [{
            "type": "text",
            "text": "Found 3 scheduled jobs for acme-hvac: [J-101, J-102, J-103]",
        }],
        "isError": False,
    },
}

example_error = {
    "jsonrpc": "2.0",
    "id": 1,
    "error": {"code": -32602, "message": "Invalid params: tenant_id missing"},
}

for label, msg in [("REQUEST", example_request), ("RESPONSE", example_response), ("ERROR", example_error)]:
    print(f"--- {label} ---")
    print(json.dumps(msg, indent=2))
    print()


---
## Section 2 — Minimal MCP Server

The server is just a JSON-RPC router with a tool registry. We implement two
methods: `tools/list` and `tools/call`. Real implementations also handle
`initialize`, `resources/list`, `prompts/list`, `ping`, etc. — they all
follow the same pattern.


In [ ]:
# JSON-RPC error codes (subset of the spec)
class MCPError:
    PARSE_ERROR      = -32700
    INVALID_REQUEST  = -32600
    METHOD_NOT_FOUND = -32601
    INVALID_PARAMS   = -32602
    INTERNAL_ERROR   = -32603
    # Application-level (anything ≤ -32000 is reserved by spec)
    UNAUTHORIZED     = -32000
    TOOL_FAILED      = -32001


@dataclass
class Tool:
    name: str
    description: str
    input_schema: dict
    handler: Callable
    requires_auth: bool = True


class MCPServer:
    """Minimal MCP server: tool registry + JSON-RPC dispatch."""

    def __init__(self, name: str):
        self.name = name
        self._tools: dict[str, Tool] = {}
        self._auth: dict[str, dict] = {}    # token -> {"tenant_id": str, "scopes": list}
        self._audit: list[dict] = []

    def register_tool(self, tool: Tool):
        self._tools[tool.name] = tool

    def register_token(self, token: str, tenant_id: str, scopes: list[str]):
        self._auth[token] = {"tenant_id": tenant_id, "scopes": scopes}

    def handle(self, raw_message: str) -> str:
        """Top-level JSON-RPC dispatcher. Returns serialized response."""
        try:
            req = json.loads(raw_message)
        except json.JSONDecodeError as e:
            return self._error_response(None, MCPError.PARSE_ERROR, str(e))

        if not isinstance(req, dict) or req.get("jsonrpc") != "2.0":
            return self._error_response(req.get("id"), MCPError.INVALID_REQUEST, "Not JSON-RPC 2.0")

        method = req.get("method")
        params = req.get("params", {})
        req_id = req.get("id")

        if method == "tools/list":
            return self._handle_list(req_id)
        elif method == "tools/call":
            return self._handle_call(req_id, params)
        else:
            return self._error_response(req_id, MCPError.METHOD_NOT_FOUND, f"Unknown method {method!r}")

    def _handle_list(self, req_id) -> str:
        tools = [{
            "name": t.name,
            "description": t.description,
            "inputSchema": t.input_schema,
        } for t in self._tools.values()]
        return json.dumps({"jsonrpc": "2.0", "id": req_id, "result": {"tools": tools}})

    def _handle_call(self, req_id, params: dict) -> str:
        tool_name = params.get("name")
        args = params.get("arguments", {}) or {}
        meta = params.get("_meta", {})
        token = meta.get("auth_token")

        if tool_name not in self._tools:
            return self._error_response(req_id, MCPError.METHOD_NOT_FOUND, f"Tool {tool_name!r} not found")

        tool = self._tools[tool_name]
        auth_ctx = None
        if tool.requires_auth:
            if not token or token not in self._auth:
                return self._error_response(req_id, MCPError.UNAUTHORIZED, "Missing or invalid auth token")
            auth_ctx = self._auth[token]
            # tenant isolation: if args include tenant_id, it MUST match the auth tenant
            if "tenant_id" in args and args["tenant_id"] != auth_ctx["tenant_id"]:
                return self._error_response(
                    req_id, MCPError.UNAUTHORIZED,
                    f"Cross-tenant access denied (token={auth_ctx['tenant_id']}, requested={args['tenant_id']})"
                )
            # auto-fill tenant_id from auth context if not provided
            args.setdefault("tenant_id", auth_ctx["tenant_id"])

        # audit log
        self._audit.append({
            "ts": time.time(),
            "tool": tool_name,
            "tenant": auth_ctx["tenant_id"] if auth_ctx else None,
            "args": {k: v for k, v in args.items() if k not in {"password", "token"}},
        })

        try:
            result = tool.handler(**args)
        except TypeError as e:
            return self._error_response(req_id, MCPError.INVALID_PARAMS, str(e))
        except Exception as e:
            return self._error_response(req_id, MCPError.TOOL_FAILED, f"{type(e).__name__}: {e}")

        return json.dumps({
            "jsonrpc": "2.0", "id": req_id,
            "result": {
                "content": [{"type": "json" if isinstance(result, (dict, list)) else "text", "data" if isinstance(result, (dict, list)) else "text": result}],
                "isError": False,
            },
        })

    def _error_response(self, req_id, code: int, message: str) -> str:
        return json.dumps({
            "jsonrpc": "2.0", "id": req_id,
            "error": {"code": code, "message": message},
        })


---
## Section 3 — ServiceTitan Tools

We expose four tools that map to real ST API surfaces:

| Tool | ST API | What it does |
|------|--------|--------------|
| `get_jobs` | `/jpm/v2/tenant/{t}/jobs` | List jobs by status/date |
| `get_technicians` | `/dispatch/v2/.../technicians` | Available techs by skill |
| `lookup_customer` | `/crm/v2/.../customers/{id}` | Single customer record |
| `get_revenue_report` | reporting API | Aggregated revenue by period |

All four use mock in-memory data so the notebook runs offline.


In [ ]:
# ── Mock ServiceTitan data ───────────────────────────────────────────────
MOCK_DB = {
    "tenants": {
        "acme-hvac":      {"name": "Acme HVAC", "plan": "pro"},
        "blue-plumbing":  {"name": "Blue Plumbing", "plan": "basic"},
    },
    "jobs": {
        "acme-hvac": [
            {"id": "J-101", "status": "scheduled", "customer_id": "C-001", "service": "ac_repair"},
            {"id": "J-102", "status": "scheduled", "customer_id": "C-002", "service": "tune_up"},
            {"id": "J-103", "status": "in_progress", "customer_id": "C-003", "service": "install"},
            {"id": "J-104", "status": "completed", "customer_id": "C-001", "service": "diagnostic"},
        ],
        "blue-plumbing": [
            {"id": "J-201", "status": "scheduled", "customer_id": "C-101", "service": "leak_repair"},
        ],
    },
    "technicians": {
        "acme-hvac": [
            {"id": "T-001", "name": "Alice", "skills": ["hvac", "electrical"], "available": True},
            {"id": "T-002", "name": "Bob",   "skills": ["hvac"],               "available": True},
            {"id": "T-003", "name": "Carol", "skills": ["plumbing"],           "available": False},
        ],
        "blue-plumbing": [
            {"id": "T-101", "name": "Dan",   "skills": ["plumbing"],           "available": True},
        ],
    },
    "customers": {
        "C-001": {"tenant": "acme-hvac", "name": "Smith Family", "tier": "vip"},
        "C-002": {"tenant": "acme-hvac", "name": "Jones Co.",    "tier": "standard"},
        "C-003": {"tenant": "acme-hvac", "name": "Lee Inc.",     "tier": "standard"},
        "C-101": {"tenant": "blue-plumbing", "name": "Park LLC", "tier": "standard"},
    },
    "revenue": {
        "acme-hvac":     {"2025-Q4": 1_250_000, "2026-Q1": 1_310_000},
        "blue-plumbing": {"2025-Q4": 410_000,   "2026-Q1": 425_000},
    },
}


# ── Tool implementations ─────────────────────────────────────────────────
def get_jobs(tenant_id: str, status: str = None, limit: int = 100):
    jobs = MOCK_DB["jobs"].get(tenant_id, [])
    if status:
        jobs = [j for j in jobs if j["status"] == status]
    return {"count": len(jobs), "jobs": jobs[:limit]}


def get_technicians(tenant_id: str, skill: str = None, available_only: bool = True):
    techs = MOCK_DB["technicians"].get(tenant_id, [])
    if skill:
        techs = [t for t in techs if skill in t["skills"]]
    if available_only:
        techs = [t for t in techs if t["available"]]
    return {"count": len(techs), "technicians": techs}


def lookup_customer(customer_id: str, tenant_id: str = None):
    cust = MOCK_DB["customers"].get(customer_id)
    if not cust:
        raise ValueError(f"Customer {customer_id!r} not found")
    if tenant_id and cust["tenant"] != tenant_id:
        raise PermissionError(f"Customer {customer_id} belongs to a different tenant")
    return cust


def get_revenue_report(tenant_id: str, period: str = None):
    revenue = MOCK_DB["revenue"].get(tenant_id, {})
    if period:
        return {"period": period, "revenue": revenue.get(period, 0)}
    return {"by_period": revenue, "total": sum(revenue.values())}


In [ ]:
# ── Register tools on the server ──────────────────────────────────────────
server = MCPServer(name="servicetitan-mcp")

server.register_tool(Tool(
    name="get_jobs",
    description="List jobs for a tenant, optionally filtered by status.",
    input_schema={
        "type": "object",
        "properties": {
            "tenant_id": {"type": "string"},
            "status": {"type": "string", "enum": ["scheduled", "in_progress", "completed"]},
            "limit": {"type": "integer", "minimum": 1, "maximum": 1000},
        },
        "required": ["tenant_id"],
    },
    handler=get_jobs,
))

server.register_tool(Tool(
    name="get_technicians",
    description="List technicians for a tenant, optionally filtered by skill and availability.",
    input_schema={
        "type": "object",
        "properties": {
            "tenant_id": {"type": "string"},
            "skill": {"type": "string"},
            "available_only": {"type": "boolean"},
        },
        "required": ["tenant_id"],
    },
    handler=get_technicians,
))

server.register_tool(Tool(
    name="lookup_customer",
    description="Get a single customer record.",
    input_schema={
        "type": "object",
        "properties": {
            "customer_id": {"type": "string"},
            "tenant_id": {"type": "string"},
        },
        "required": ["customer_id"],
    },
    handler=lookup_customer,
))

server.register_tool(Tool(
    name="get_revenue_report",
    description="Aggregated revenue report for a tenant.",
    input_schema={
        "type": "object",
        "properties": {
            "tenant_id": {"type": "string"},
            "period": {"type": "string"},
        },
        "required": ["tenant_id"],
    },
    handler=get_revenue_report,
))

# ── Register a tenant token ───────────────────────────────────────────────
server.register_token("acme-token-xyz", tenant_id="acme-hvac", scopes=["read"])
server.register_token("blue-token-abc", tenant_id="blue-plumbing", scopes=["read"])

print(f"Server '{server.name}' ready.  Tools registered: {list(server._tools)}")
print(f"  Tenants configured: {list(set(v['tenant_id'] for v in server._auth.values()))}")


---
## Section 4 — Tool Calls End-to-End

Walk through what an LLM-driven client would actually send.


In [ ]:
def call(method: str, params: dict, token: str = None, req_id: int = None) -> dict:
    """Helper to send a JSON-RPC request and parse the response."""
    if req_id is None:
        req_id = uuid.uuid4().int % 10_000
    if token:
        params.setdefault("_meta", {})["auth_token"] = token
    req = {"jsonrpc": "2.0", "id": req_id, "method": method, "params": params}
    raw_resp = server.handle(json.dumps(req))
    return json.loads(raw_resp)


# 1. list available tools
resp = call("tools/list", {}, token=None)
print("→ tools/list")
print(f"  got {len(resp['result']['tools'])} tools: {[t['name'] for t in resp['result']['tools']]}")

# 2. happy path: get scheduled jobs for acme
resp = call("tools/call", {"name": "get_jobs", "arguments": {"status": "scheduled"}}, token="acme-token-xyz")
print(f"\n→ get_jobs(status=scheduled) with acme token")
print(f"  result: {resp['result']}")

# 3. happy path: revenue report
resp = call("tools/call", {"name": "get_revenue_report", "arguments": {}}, token="acme-token-xyz")
print(f"\n→ get_revenue_report() with acme token")
print(f"  result: {resp['result']}")

# 4. unauthorized: no token
resp = call("tools/call", {"name": "get_jobs", "arguments": {"tenant_id": "acme-hvac"}}, token=None)
print(f"\n→ get_jobs() with NO token (expected fail)")
print(f"  error: {resp.get('error')}")

# 5. cross-tenant attempt: blue token trying to read acme data
resp = call("tools/call",
    {"name": "get_jobs", "arguments": {"tenant_id": "acme-hvac"}},
    token="blue-token-abc")
print(f"\n→ blue token trying to read acme data (expected fail)")
print(f"  error: {resp.get('error')}")

# 6. cross-tenant via lookup_customer
resp = call("tools/call",
    {"name": "lookup_customer", "arguments": {"customer_id": "C-001"}},
    token="blue-token-abc")
print(f"\n→ blue token looking up acme customer C-001 (expected fail)")
print(f"  error: {resp.get('error')}")
print(f"  result: {resp.get('result')}")


---
## Section 5 — Multi-Tenant Isolation and Audit

Two things every production MCP server **must** have:

1. **Tenant isolation**: a token grants access to exactly one tenant. The server
   enforces this — never trust the client to send the right `tenant_id`. We
   auto-fill from the auth context and reject mismatches.

2. **Audit log**: every tool call is logged with `(timestamp, tool, tenant,
   args)`. This is non-negotiable for any regulated industry — and ST has SOC2,
   so the platform team will demand it.


In [ ]:
print(f"Audit log: {len(server._audit)} entries\n")
for entry in server._audit:
    print(f"  {time.strftime('%H:%M:%S', time.localtime(entry['ts']))}  "
          f"{entry['tool']:<22} tenant={entry['tenant'] or '-':<14}  args={entry['args']}")


---
## Section 6 — Wiring to a LangGraph Agent

Notebook 11 builds Atlas as a LangGraph StateGraph. To plug this MCP server in,
we wrap the JSON-RPC calls as Python functions that the LLM can call directly.

In a production deployment:
- The LLM is told about the tools via the system prompt (built from `tools/list`).
- When the LLM decides to call a tool, the agent loop catches the structured
  tool-use block, forwards it as `tools/call` to the MCP server, and feeds the
  result back to the LLM for the next turn.

The Python-side wrapper layer below is tiny — about 20 lines.


In [ ]:
class MCPClient:
    """Pythonic wrapper around an MCP server for use inside an agent."""

    def __init__(self, server: MCPServer, token: str):
        self.server = server
        self.token = token

    def list_tools(self) -> list[dict]:
        resp = json.loads(self.server.handle(json.dumps(
            {"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {}}
        )))
        return resp["result"]["tools"]

    def call(self, tool_name: str, **arguments) -> Any:
        req = {
            "jsonrpc": "2.0", "id": 1, "method": "tools/call",
            "params": {
                "name": tool_name, "arguments": arguments,
                "_meta": {"auth_token": self.token},
            },
        }
        resp = json.loads(self.server.handle(json.dumps(req)))
        if "error" in resp:
            raise RuntimeError(f"MCP call failed: {resp['error']['message']}")
        return resp["result"]["content"][0].get("data") or resp["result"]["content"][0].get("text")


# ── Simulated agent loop ──────────────────────────────────────────────────
client = MCPClient(server, token="acme-token-xyz")

# Pretend the LLM emitted these tool calls in sequence to answer:
# "How many scheduled jobs does Acme have, and what's the customer info for job J-101?"
print("--- Simulated agent loop ---\n")

# Turn 1: LLM calls get_jobs
print("Turn 1: LLM calls get_jobs(status='scheduled')")
jobs_result = client.call("get_jobs", status="scheduled")
print(f"  → {jobs_result}\n")

# Turn 2: LLM sees J-101 and calls lookup_customer for C-001
print("Turn 2: LLM calls lookup_customer(customer_id='C-001')")
cust_result = client.call("lookup_customer", customer_id="C-001")
print(f"  → {cust_result}\n")

# Turn 3: LLM produces final answer
print("Turn 3: LLM final answer:")
print(f'  "Acme HVAC has {jobs_result["count"]} scheduled jobs. Job J-101 belongs to')
print(f'   {cust_result["name"]}, a {cust_result["tier"]} tier customer."')


---
## Section 7 — Production Checklist

What we built is the **mechanics**. To deploy a real MCP server at ST scale,
you'd add:

### Transport
- For local dev (Claude Desktop): stdio transport — process spawns subprocess, JSON-RPC over pipes.
- For remote (Atlas in browser): HTTP+SSE transport — server-sent events for streaming, POST for requests.
- For service mesh: gRPC bridge wrapping the JSON-RPC. ST's stack is .NET, so the MCP server itself would be a Python sidecar OR a port to C# (Anthropic publishes a TypeScript SDK; a C# port is straightforward).

### Auth
- Replace the dict-based tokens with **OAuth 2.0** integration to ST's existing auth service.
- Each ST app key already maps to a tenant — the MCP server should reuse that mapping.
- Scopes: read-only (`get_*`) vs write (`assign_*`, `dispatch_*`). The Atlas LLM is read-only by default; write actions require a human-in-the-loop confirmation step.

### Observability
- Prometheus metrics: `mcp_tool_calls_total{tool, tenant, status}`, `mcp_tool_latency_seconds`.
- Structured logs: every audit entry to a SIEM-readable JSON log.
- Trace propagation: pass `traceparent` header through to underlying ST APIs.

### Rate limits
- Per-tenant token bucket (use notebook 27's `SlidingWindowRateLimiter`).
- Per-tool quotas — `get_revenue_report` is expensive (warehouse query); should be quota-limited per tenant per hour.

### Caching
- LLM agents often call the same tool multiple times in a single conversation (Atlas keeps re-checking job status). A small LRU cache (see notebooks 27/28) in front of expensive tools is a big win — 60-80% hit rate typical.

### Sandboxing
- Output size limits: a tool that returns 10MB of data will eat your context window. Truncate responses to a few KB with a "use cursor X for more" pattern.

### The MCP Python SDK

For production, swap our hand-rolled implementation for:

```python
# pip install mcp
from mcp.server import Server, NotificationOptions
from mcp.server.stdio import stdio_server
import mcp.types as types

server = Server("servicetitan-mcp")

@server.list_tools()
async def list_tools() -> list[types.Tool]:
    return [types.Tool(name="get_jobs", description="...", inputSchema={...})]

@server.call_tool()
async def call_tool(name: str, arguments: dict) -> list[types.TextContent]:
    if name == "get_jobs":
        result = get_jobs(**arguments)
        return [types.TextContent(type="text", text=json.dumps(result))]
```

The SDK handles transport, framing, and reconnection. The protocol logic — tool
registry, schema validation, auth, audit — is the same as what we built above.


In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────
print("=" * 60)
print("Notebook 24 — MCP Server for ServiceTitan Tools")
print("=" * 60)
print(f"  Section 1 — protocol primer:        JSON-RPC 2.0 documented")
print(f"  Section 2 — minimal MCP server:     dispatch + registry built")
print(f"  Section 3 — tools registered:       {list(server._tools)}")
print(f"  Section 4 — end-to-end calls:       6 demos (auth + cross-tenant)")
print(f"  Section 5 — audit log entries:      {len(server._audit)}")
print(f"  Section 6 — agent loop:             simulated 3-turn Atlas flow")
print(f"  Section 7 — production checklist:   documented")
print("=" * 60)
